# Extra rollouts (indices 5-9) for the 309 eligible training-pool questions

Generates 5 *more* rollouts per question on top of the 5 that already exist (experiment 001), to test whether the oracle ceiling keeps climbing with more attempts -- see `scripts/evaluation/oracle_vs_rollout_count.py`: with the existing 5 rollouts the curve was still rising steeply (17.8% -> 53.4%, +5.9pp from the 4th to the 5th rollout, no sign of flattening).

No Kaggle secrets needed: the repo and the CharXiv HF dataset are both public.


In [ ]:
!git clone https://github.com/yahorlahunovich/chart-prm.git
%cd chart-prm
!pip install -q transformers accelerate bitsandbytes datasets qwen-vl-utils pillow torchvision


In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info
import json
import os

# Load Model in 4-bit precision to fit within the 16GB VRAM of a T4 GPU
model_id = "Qwen/Qwen2.5-VL-3B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=quantization_config
)

processor = AutoProcessor.from_pretrained(model_id)


In [ ]:
from datasets import load_dataset
import sys
sys.path.append("src")
from chart_prm.generator import build_generation_prompt
import json

print("Downloading CharXiv...")
dataset = load_dataset("princeton-nlp/CharXiv", split="validation")

# The 309 already-eligible training-pool questions (experiments/008, /011) --
# embedded directly so this notebook doesn't depend on branch/commit state.
target_ids = set(["1002", "1008", "1009", "1011", "1015", "1022", "1028", "1031", "1036", "1038", "1044", "1051", "1071", "108", "1081", "1122", "1123", "1126", "113", "1138", "1144", "1146", "1147", "1152", "1153", "1160", "1170", "1174", "1177", "1183", "1186", "1188", "1192", "1196", "1208", "1212", "1219", "1222", "1223", "1225", "1231", "1233", "1235", "1237", "1253", "1258", "1268", "1275", "1281", "130", "1314", "1317", "1322", "1329", "1330", "1332", "1336", "1343", "1348", "135", "1352", "1360", "137", "1373", "1374", "1375", "1378", "1397", "14", "1402", "1406", "1413", "1425", "1432", "1438", "145", "1455", "1463", "1501", "1511", "1514", "1520", "153", "1531", "1555", "1569", "157", "1573", "1580", "1585", "1588", "159", "160", "1607", "1608", "1611", "1653", "1654", "1658", "1661", "1664", "1671", "1672", "1677", "1680", "1687", "1704", "1725", "1739", "1762", "179", "1793", "180", "1803", "181", "1815", "1822", "1827", "183", "1837", "1841", "1844", "1856", "1860", "1865", "1868", "1887", "1898", "1901", "1913", "1914", "1916", "1921", "1929", "1932", "1937", "1942", "1968", "1969", "1971", "1973", "1976", "1980", "1986", "1987", "1994", "20", "2000", "2003", "2004", "2010", "2030", "2055", "2056", "2061", "2083", "2097", "210", "212", "2130", "2133", "2140", "2145", "2147", "215", "2158", "2166", "2188", "2203", "2208", "2211", "2234", "2236", "2238", "2240", "225", "2271", "2274", "2287", "2307", "2314", "2317", "2329", "2338", "2346", "2347", "2361", "2362", "2372", "243", "247", "248", "268", "279", "286", "288", "29", "291", "292", "300", "330", "337", "341", "354", "36", "384", "391", "394", "395", "40", "407", "411", "416", "418", "421", "423", "447", "450", "461", "462", "464", "466", "477", "480", "488", "489", "49", "493", "495", "5", "506", "507", "509", "51", "510", "514", "517", "525", "53", "543", "548", "550", "555", "568", "575", "586", "588", "593", "598", "602", "61", "625", "628", "629", "634", "636", "639", "649", "65", "650", "661", "68", "697", "703", "706", "708", "711", "722", "727", "741", "749", "75", "756", "757", "764", "766", "770", "771", "782", "787", "794", "8", "810", "833", "84", "840", "844", "856", "861", "880", "882", "890", "897", "900", "917", "92", "920", "929", "930", "945", "955", "96", "962", "965", "966", "970", "971", "985", "987"])
assert len(target_ids) == 309

with open("data/CharXiv/data/reasoning_val.json", "r") as f:
    all_keys = list(json.load(f).keys())

# Filter dataset to only the target questions (same technique as model_inference.ipynb)
dataset = dataset.filter(lambda x, idx: all_keys[idx] in target_ids, with_indices=True)

# Position-in-filtered-dataset -> true figure_id, computed locally so question_id can
# be written correctly straight into the output (no post-hoc fix_jsonl_ids.py needed).
ordered_target_keys = [k for k in all_keys if k in target_ids]
index_to_figure_id = {i: k for i, k in enumerate(ordered_target_keys)}

num_samples = len(dataset)
print(f"{num_samples} questions loaded (expected 309)")
assert num_samples == 309


In [ ]:
output_file = "generated_extra_rollouts.jsonl"
save_every = 5
ROLLOUT_START = 5    # the original 5 rollouts are indices 0-4; these are new
NUM_NEW_ROLLOUTS = 5  # generates rollout_index 5..9

start_index = 0
if os.path.exists(output_file):
    with open(output_file, "r", encoding="utf-8") as f:
        total_lines = sum(1 for _ in f)
        start_index = total_lines // NUM_NEW_ROLLOUTS
    print(f"Found existing checkpoint with {total_lines} trajectories. Resuming from sample {start_index}")

for i, sample in enumerate(dataset):
    if i < start_index:
        continue

    image = sample['image']
    question = sample["reasoning_q"]

    prompt_text = build_generation_prompt(question)

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt_text},
            ],
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to("cuda")

    # Generate sequentially to avoid VRAM OOM, same as the original pipeline
    for offset in range(NUM_NEW_ROLLOUTS):
        rollout_idx = ROLLOUT_START + offset
        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=True,
                temperature=0.7
            )

        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]

        res = {
            "sample_index": i,
            "question_id": index_to_figure_id[i],
            "rollout_index": rollout_idx,
            "question": question,
            "model_output": output_text,
        }
        if "reasoning_a" in sample: res["ground_truth"] = sample["reasoning_a"]
        if "answer" in sample: res["ground_truth"] = sample["answer"]

        with open(output_file, "a", encoding="utf-8") as f:
            f.write(json.dumps(res, ensure_ascii=False) + "\n")

    if (i + 1) % save_every == 0 or (i + 1) == num_samples:
        print(f"Processed {i+1}/{num_samples} questions ({NUM_NEW_ROLLOUTS} new rollouts each)...")

print(f"Finished generating {num_samples * NUM_NEW_ROLLOUTS} new trajectories!")
